## 1. Import Libraries

In this step, we import the necessary libraries for data analysis, visualization, and preprocessing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import missingno
from sklearn.preprocessing import OneHotEncoder , LabelEncoder ,OrdinalEncoder ,  StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
# from sqlalchemy import Transaction

## 2. Load Dataset

Here, we load the dataset from a CSV file using pandas.

In [ ]:
data = pd.read_csv(r"car_price.csv")

In [ ]:
df = data.copy()
df

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
missingno.matrix(df)

In [ ]:
print(df.shape)
print(df.loc[df.isna().sum(axis=1)>2].shape)
print(df.loc[df.isna().sum(axis=1)>1].shape)
print(df[~(df.isna().sum(axis=1)>2)].shape)

In [ ]:
df = df[~(df.isna().sum(axis=1)>2)]
print(df.shape)
df.head(3)

In [ ]:
print("categorical summary of the dataset:")
(df.select_dtypes(include='object').describe())

In [ ]:
df['year'].value_counts()

In [ ]:
for i in df.columns:
    nunique_ = df[i].nunique()
    head_ = df[i].value_counts().sort_values(ascending=False).head(5)
    tail_ = df[i].value_counts().sort_values(ascending=False).tail(5)

    print(f'{i}\n')
    print(f'nunique : {nunique_}\n')
    print(f'head 5 value_count : {head_}\n')
    print(f'tail 5 value_count : {tail_}\n')
    print('*'*50)
    print()



## 3. Exploratory Data Analysis (EDA)

We explore the dataset to understand its structure, data types, and basic statistical information.

In [ ]:
print("The shape of the dataset is:",df.shape)
print()
print("Data types of each column:")
print(df.info())
print()
print("Statistical summary of the dataset:")
print(df.describe())
print()
print()
print("Number of missing values in each column:")
print(df.isnull().sum())
print()
print("Number of duplicate rows:")
print(df.duplicated().sum())
print()


## we found in the last step that there are 366 duplicate rows
we will drop them and check if there are any duplicate values

In [ ]:
df.drop_duplicates(inplace=True)
print("Number of duplicate rows after dropping duplicates:")
print(df.duplicated().sum())
print()

## check for missing values
the first step we will drop the row if all values are missing

In [ ]:
# df[df.isna().sum(axis=1)>0]
(df[df.isna().any(axis=1)])

In [ ]:
# df[df.isna().sum(axis=1)>len(df.columns)]
(df[df.isna().all(axis=1)])

# not exist row have all feature is null

In [ ]:
# df.dropna(how='all', inplace=True)

## 7. Separating Numerical and Categorical Columns

We split the dataset columns into:
- Numerical features
- Categorical features

In [ ]:
num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(include="object").columns
print("Numerical columns:", num_cols.tolist())
print("Categorical columns:", cat_cols.tolist())
print()

## 8. Re-check Missing Values

We inspect missing values again before applying imputation methods.

In [ ]:

print("Number of missing values in each column:")
print(df.isnull().sum())
print()

## 9. Handling Missing Values in Categorical Features

We fill missing values in categorical columns using the most frequent value (mode).

In [ ]:
df[['Make', 'model', 'transmission', 'fuelType']]

 خد بالك في مسافات في اسامي الكلاسات

 في الاول نضمن ان الداتا بتاعتي الكلاسات الي في كل عمود من غير اي مسافات
 * we will use => : `strip()`

كمان ممكن يكون في كلاسات زي بعض بس الفرق انه الحروف صغيرة او كبيرة

 * use =>> : `lower()` , `upper()`

In [ ]:
df[df['model']=='A1']

In [ ]:
df[df['model'].str.strip().str.upper() =='A1']

In [ ]:
# df['model'].str.findall(r'[A-Za-z0-9]+')
# df['model'].str.extract(r'([A-Za-z0-9]+)')

* `contain` & `match` cant treat with nan

In [ ]:
# print(df['model'].str.contains(r'[A-Za-z0-9]+'))
# print(df['model'].str.match(r'^[A-Za-z0-9]+$'))

In [ ]:
print(df['model'].str.contains(r'[A-Za-z0-9]+', na=False))
print(df['model'].str.match(r'^[A-Za-z0-9]+$', na=False))

## Solve

In [ ]:
print(df['model'].str.extract(r'([A-Za-z0-9]+)')[0].str.capitalize())

In [ ]:
for i in df.select_dtypes(include='object').columns:
    df[i] = df[i].str.extract(r'([A-Za-z0-9]+)')
    df[i] = df[i].str.capitalize()

In [ ]:
df.head(3)

## correlation about all columns by clean data

In [ ]:
df2 = df.copy()

In [ ]:
df2.shape

In [ ]:
df.isna().sum().sum()

In [ ]:
df2 = df2.dropna()

In [ ]:
df2.head(3)

In [ ]:
transform_num_col = Pipeline(
    steps=[
        ('scale' , StandardScaler())
    ]
)

# transform_cat_col_OH = Pipeline(
#     steps=[
#         ('one_hot' , OneHotEncoder(sparse_output=False))
#     ]
# )

transform_cat_num_label = Pipeline(
    steps=[
        ('label' , OrdinalEncoder())
    ]
)

preprosecor = ColumnTransformer(
    transformers=[
        ('num' , transform_num_col , ['mileage' , 'tax' , 'mpg' , 'engineSize']),
        # ('cat_OH' , transform_cat_col_OH , ['transmission' , 'fuelType']),
        ('cat_label' , transform_cat_num_label , ['model','transmission' , 'fuelType'])
    ],remainder='passthrough'
)

preprosecor


In [ ]:
df_transformed= preprosecor.fit_transform(df2)
df_transformed

In [ ]:
df_transformed_df = pd.DataFrame(df_transformed , columns=preprosecor.get_feature_names_out())
df_transformed_df

In [ ]:
mapping = df_transformed_df.groupby('remainder__Make')['remainder__price'].mean().to_dict()
mapping

In [ ]:
df_transformed_df['remainder__Make'] = df_transformed_df['remainder__Make'].map(mapping)

In [ ]:
df_transformed_df.head(5)

In [ ]:
df_transformed_df.corr()

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df_transformed_df.corr() , annot=True , fmt='.2g' , cmap='Blues')

## handle `Make`

In [ ]:
df[df['model'] == 'A1']

In [ ]:
print(df[df['model'] == 'A1']['Make'].unique())

In [ ]:
df.groupby('model')['Make'].value_counts(dropna=False)

In [ ]:
group = df.groupby('model')['Make']

for cat , g in group:
    print(f'{cat}\n')
    print(f'{g}\n')
    print('*'*50)

In [ ]:
df['Make'] = df.groupby('model')['Make'].ffill().bfill()

In [ ]:
df.isna().sum()

# handel `model`

In [ ]:
group = df.groupby(['transmission','Make' , 'fuelType' ])[['model']]
for cat , g in group:
    print(f'{cat}\n')
    print(f'{g}\n')
    print('*'*50)

In [ ]:
group = (df.groupby(['transmission', 'Make' ,'fuelType'])[['model']]
        .value_counts(dropna=False)
        .to_frame()
        )

group.head(20)

In [ ]:
group = group.reset_index(level=[0,1,2])
group

In [ ]:
group = (group.groupby(['transmission' , 'Make' , 'fuelType'])['count']
        .idxmax()
        .to_frame()
        .reset_index())
group

* basic

In [ ]:
group = (df.groupby(['transmission', 'Make' ,'fuelType'])[['model']]
        .value_counts(dropna=False)
        .to_frame()
        .reset_index(level=[0,1,2])
        .groupby(['transmission' , 'Make' , 'fuelType'])['count']
        .idxmax()
        .reset_index()
        )

group

In [ ]:
mask_group = (group['transmission'] == 'Automatic') & (group['Make'] == 'Audi') & (group['fuelType'] == 'Petrol')
fill_array = group[mask_group]['count']
fill_array.iloc[0]

In [ ]:
df[(df['transmission']=='Automatic') & (df['Make'] == 'Audi') & (df['fuelType'] == 'Diesel')]['model'].isna().sum()

In [ ]:
# mask = ((df['transmission']=='Automatic') & (df['Make'] == 'Audi') & (df['fuelType'] == 'Diesel'))
# df.loc[mask , 'model'] = df.loc[mask , 'model'].fillna('A6')

In [ ]:
for trans in df['transmission'].unique():
    for make in df['Make'].unique():
        for fuel in df['fuelType'].unique():
            
            mask_group = (group['transmission'] == trans) & (group['Make'] == make) & (group['fuelType'] == fuel)
            fill_array = group[mask_group]['count']

            if fill_array.shape[0] > 0 :
                fill_= fill_array.iloc[0]

                mask_df = ((df['transmission']==trans) & (df['Make'] == make) & (df['fuelType'] == fuel))
                df.loc[mask_df , 'model'] = df.loc[mask_df , 'model'].fillna(fill_)


In [ ]:
df[(df['transmission']=='Automatic') & (df['Make'] == 'Audi') & (df['fuelType'] == 'Diesel')]['model'].isna().sum()

In [ ]:
df.isna().sum()

In [ ]:
df[df['model'].isna()]

In [ ]:
group

In [ ]:
group.set_index(['transmission' , 'Make' , 'fuelType'])

In [ ]:
mapping = group.set_index(['transmission' , 'Make' , 'fuelType'])['count'].to_dict()
mapping

In [ ]:
df.set_index(['transmission' , 'Make' , 'fuelType'])['model']

In [ ]:
df.set_index(['transmission' , 'Make' , 'fuelType']).index

In [ ]:
df.loc[:, 'model'] = df['model'].fillna(
    pd.Series(
        df.set_index(['transmission','Make','fuelType']).index.map(mapping)
    )
)

In [ ]:
df[(df['transmission']=='Automatic') & (df['Make'] == 'Audi') & (df['fuelType'] == 'Diesel')]['model'].isna().sum()

In [ ]:
df.isna().sum()

In [ ]:
df.dropna(subset='model' , inplace=True)

In [ ]:
df.isna().sum()

## Handle Year

In [ ]:
group = (df.groupby(['Make' , 'model'])['year']
            .value_counts(dropna=False)
            .to_frame()
            .head(20))
group

In [ ]:
group = (df.groupby(['Make' , 'model'])['year']
            .value_counts()
            .to_frame()
            .reset_index(level=[0,1])
            .groupby(['Make','model'])['count']
            .idxmax()
            .to_frame()
            .reset_index()
            .rename(columns={'count' : 'year'}))

group

In [ ]:
mask_group = (group['Make'] == 'Audi') & (group['model'] == 'A1')
fill_array = group[mask_group]['year']
fill_array.iloc[0]

In [ ]:
df[(df['Make']=='Audi') & (df['model'] == 'A1')]['year'].isna().sum()

In [ ]:
# mask = ((df['Make']=='Audi') & (df['model'] == 'A1'))
# df.loc[mask , 'year'] = df.loc[mask , 'year'].fillna('2016.0')

In [ ]:
df[(df['Make']=='Audi') & (df['model'] == 'A1')]['year'].isna().sum()

In [ ]:

for make in df['Make'].unique():
    for model in df['model'].unique():
        
        mask_group = (group['Make'] == make) & (group['model'] == model)
        fill_array = group[mask_group]['year']

        if fill_array.shape[0] > 0 :
            fill_= fill_array.iloc[0]

            mask_df = ((df['Make']==make) & (df['model'] == model))
            df.loc[mask_df , 'year'] = df.loc[mask_df , 'year'].fillna(fill_)


In [ ]:
df[(df['Make']=='Audi') & (df['model'] == 'A1')]['year'].isna().sum()

In [ ]:
df.isna().sum()

In [ ]:
group

In [ ]:
mapping = group.set_index(['Make' , 'model'])['year'].to_dict()
mapping

In [ ]:
df.set_index(['Make' , 'model']).index

In [ ]:
df.loc[:, 'year'] = df['year'].fillna(
    pd.Series(
        df.set_index(['Make' , 'model']).index.map(mapping)
    )
)

In [ ]:
df[(df['Make']=='Audi') & (df['model'] == 'A1')]['year'].isna().sum()

In [ ]:
df.isna().sum()

In [ ]:
df['year'] = df['year'].astype(int)
# df['year'] = pd.to_datetime(df['year'], format='%Y')

لما عملت لوب بس مع المودل طلع ان في 380 نالز وده بسبب ان اكيد كان في قيمة من القيم الي الوب بتلف عليهم كانت نال

ولما عملت مابينج طلع عندك 350 نالز وده بسسب ان كان في بعض المفاتيح الي انت بتبحث علي اساسها كانت نالز فكان بيرجع نالز حسب الفانكشن تبدلها ولا لا

بس لما عملنا الاتنين اتبقا 42 نالز ومفرقش عملت مين الاول

فا احنا هنعمل ماب عشان اسرع وبعد كدة نعمل لوب عشان تكون الماب قللت النالز فتقلل معانا عدد اللوب

كل ده حصل عشان انا كنت سامح ان ممكن عادي صف يكون فيه اكتر من اتنين نالز

print(df.shape)

print(df.loc[df.isna().sum(axis=1)>2].shape)

print(df.loc[df.isna().sum(axis=1)>1].shape)

print(df[~(df.isna().sum(axis=1)>2)].shape)

(72069, 10)

(507, 10)

(4405, 10)

(71562, 10)

In [ ]:
# print(df.shape)
# print(df.loc[df.isna().sum(axis=1)>2].shape)
# print(df.loc[df.isna().sum(axis=1)>1].shape)
# print(df[~(df.isna().sum(axis=1)>2)].shape)

In [ ]:
df.head(3)

# Handle FuelType

In [ ]:
group = (df.groupby('model')['fuelType']
        .value_counts()
        .to_frame()
        .reset_index('model')
        .groupby(['model'])['count']
        .idxmax()
        .to_frame()
        .reset_index()
)
group

In [ ]:
df.isna().sum()

In [ ]:
for model in df['model'].unique():
    
    mask_group = (group['model'] == model)
    fill_array = group[mask_group]['count']

    if fill_array.shape[0] > 0 :
        fill_= fill_array.iloc[0]

        mask_df =(df['model'] == model)& (df['fuelType'].isna())
        df.loc[mask_df , 'fuelType'] = df.loc[mask_df , 'fuelType'].fillna(fill_)


In [ ]:
df.isna().sum()

In [ ]:
mapping = group.set_index('model')['count'].to_dict()
mapping

In [ ]:
df.loc[: , 'fuelType'] = df['fuelType'].fillna(
        df['model'].map(mapping)
)

In [ ]:
df.isna().sum()

In [ ]:
df.corr(numeric_only=True)

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(numeric_only=True) , annot=True , fmt='.2g' , cmap='Blues')

# handle EngineSize

In [ ]:
df.head(3)

In [ ]:
group = df.groupby(['model' , 'fuelType'])['engineSize'].value_counts().to_frame()
group.head(12)

In [ ]:
group = group.reset_index(level=[0,1]).groupby(['model' , 'fuelType'])['count'].idxmax().to_frame().reset_index()
group

In [ ]:
for model in df['model'].unique():
    for fuel in df['fuelType'].unique():
        mask_group = (group['fuelType'] == fuel) & (group['model'] == model)
        fuel_array = group[mask_group ]['count']

        if fuel_array.shape[0] > 0 :
            fill_ = fuel_array.iloc[0]

            mask_df = (df['model'] == model) & (df['fuelType'] == fuel) & (df['engineSize'].isna())
            df.loc[mask_df , 'engineSize'] = df.loc[mask_df , 'engineSize'].fillna(fill_)

In [ ]:
df.isna().sum()

In [ ]:
group

In [ ]:
mapping = group.set_index(['model','fuelType'])['count'].to_dict()
mapping

In [ ]:
df['engineSize'] = df['engineSize'].fillna(
    pd.Series(
        df.set_index(['model' , 'fuelType']).index.map(mapping)
    )
)

In [ ]:
df.isna().sum()

## handle transimision

In [ ]:
df.head()

In [ ]:
mode_model = df.groupby('model')['transmission'].value_counts().to_frame().reset_index(['model']).groupby('model').idxmax().reset_index()
mode_model.head(12)

In [ ]:
fill_array = mode_model[mode_model['model']=='7']
fill_array['count'].iloc[0]

In [ ]:
for model in df['model'].unique():
    mask = (df['model'] == model) & (df['transmission'].isna())

    fill_array = mode_model[mode_model['model']==model]

    if fill_array.shape[0] > 0:
        fill_ = fill_array['count'].iloc[0]
        df.loc[mask , 'transmission'] = fill_

In [ ]:
df.isna().sum()

In [ ]:
mode_model = df.groupby('model')['transmission'].value_counts().to_frame().reset_index(['model'])
mode_model.head(12)

In [ ]:
mode_model = df.groupby('model')['transmission'].value_counts().to_frame().reset_index(['model']).groupby('model').idxmax()['count'].to_dict()
mode_model

In [ ]:
df.loc[: ,'transmission'] = df['transmission'].fillna(
    pd.Series(
        df.set_index('model').index.map(mode_model)
    )
)

In [ ]:
df.isna().sum()

# handling `tax` , `mpg` , `mileage`

In [ ]:
df.head(3)

# tax

In [ ]:
mapping = df.groupby('Make')['tax'].mean().to_dict()
mapping

In [ ]:
df['Make'].head()

In [ ]:
df.loc[: ,'tax'] = df['tax'].fillna(df['Make'].map(mapping))

# mileage

In [ ]:
# mapping = df.groupby('Make')['mileage'].mean().to_dict()
# mapping

In [ ]:
mapping = df.groupby('Make')['mileage'].median().to_dict()
mapping

In [ ]:
df.loc[: ,'mileage'] = df['mileage'].fillna(df['Make'].map(mapping))

# mpg

In [ ]:
# mapping = df.groupby('Make')['mpg'].median().to_dict()
# mapping

In [ ]:
mapping = df.groupby('Make')['mpg'].mean().to_dict()
mapping

In [ ]:
df.loc[: ,'mpg'] = df['mpg'].fillna(df['Make'].map(mapping))

In [ ]:
df.isna().sum()

# price

In [ ]:
df.head()

In [ ]:
df.select_dtypes(include='object').nunique()

In [ ]:
# num_col = Pipeline(
#     steps=[
#         ('scale' , StandardScaler())
#     ]
# )

# cat_col_OH = Pipeline(
#     steps=[
#         ('one_hot' , OneHotEncoder(sparse_output=False , handle_unknown="ignore"))
#     ]
# )

# cat_col_ordinal = Pipeline(
#     steps=[
#         ('ordinal',OrdinalEncoder())
#     ]
# )

preproccesor_linear = ColumnTransformer(
    transformers=[
        ('num' ,  StandardScaler() , ['year' , 'mileage' , 'tax' , 'mpg' , 'engineSize']),
        ('cat_OH' , OneHotEncoder(sparse_output=False , handle_unknown="ignore") , ['fuelType' , 'transmission']),
        ('cat_ordinal' , OrdinalEncoder() , ['model'])
    ],remainder='passthrough'
)

before_model = Pipeline(
    steps=[
        ('preproccesor' , preproccesor_linear)
    ]
)
before_model


In [ ]:
target_encode = df.groupby('Make')['price'].mean().to_dict()
target_encode

In [ ]:
df['Make'] = df['Make'].map(target_encode)

In [ ]:
df.head()

In [ ]:
df_transform = before_model.fit_transform(df)
df_transform

In [ ]:
df_transform_df = pd.DataFrame(df_transform , columns=before_model.get_feature_names_out())
df_transform_df

In [ ]:
from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors=2)
df_imputed = imputer.fit_transform(df_transform_df)
df_imputed

In [ ]:
df_transform_df = pd.DataFrame(df_imputed , columns = imputer.get_feature_names_out())
df_transform_df

In [ ]:
df_transform_df.isna().sum()

In [ ]:
# cat_imputer = SimpleImputer(strategy='most_frequent')
# df[['transmission']] = cat_imputer.fit_transform(
#     df[['transmission']]
# )


## 10. Handling Missing Values in Numerical Features

We fill missing values in numerical columns using the median value.

In [ ]:
df_transform_df.head()

In [ ]:
# num_imputer = SimpleImputer(strategy='median')
# df[['tax', 'mpg', 'mileage']] = num_imputer.fit_transform(
#     df[[ 'tax', 'mpg', 'mileage']]
# )

## 11. Final Data Check

We verify that there are no missing values remaining after cleaning for any column without the price column.

In [ ]:
print("Number of missing values in each column:")
print(df_transform_df.isnull().sum())
print()

## 12. Save Cleaned Dataset

Finally, we save the cleaned dataset into a new CSV file for future use.

In [ ]:
# df_transform_df.to_csv(r"car_price_cleaned_2.csv", index=False)

In [ ]:
def detect_outliers(df):
    
    outlier_summary = {}


    for col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]

        if len(outliers) > 0:
            outlier_summary[col] = {
                "count": len(outliers),
                "lower_bound": lower_bound,
                "upper_bound": upper_bound
            }

    return outlier_summary

In [ ]:
def view_outliers_with_values(df):
    
    outlier_data = {}


    for col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outliers = df[(df[col] < lower) | (df[col] > upper)]

        if not outliers.empty:
            outlier_data[col] = outliers[col]

    return outlier_data

In [ ]:
df_transform_df

In [ ]:
feature_named = before_model.named_steps['preproccesor'].get_feature_names_out()

cat_cols = [col for col in feature_named if 'cat_OH' in col or 'cat_ordinal' in col or 'remainder' in col]
cat_cols

In [ ]:
detect= detect_outliers(df_transform_df.drop(columns=cat_cols))
detect

In [ ]:
detect= view_outliers_with_values(df_transform_df.drop(columns=cat_cols))
detect

In [ ]:
cols = [i for i in detect]
cols

In [ ]:
for i in cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(data=df_transform_df , x = i)
    plt.show()

In [ ]:
def remove_outliers_with_values(df):

    for col in df_no_outlairs.columns:
        Q1 = df_no_outlairs[col].quantile(0.25)
        Q3 = df_no_outlairs[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        df_no_outlairs =df_no_outlairs[(df[col] >= lower) & (df_no_outlairs[col] <= upper)]